# Support Vector Machines (SVM) with Various Kernels

This notebook covers Support Vector Machines including Linear SVM, and various kernel methods (Polynomial, RBF, Sigmoid) with detailed examples using PyTorch and scikit-learn.

## 1. Import Required Libraries

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_circles, make_moons, load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
from sklearn.inspection import DecisionBoundaryDisplay
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch Version: {torch.__version__}')

## 2. SVM Theory and Mathematical Foundation

### Objective
Find the hyperplane that maximizes the margin between classes

### Linear SVM
Decision function: $f(x) = w^T x + b$

Optimization problem:
$$\min_{w,b} \frac{1}{2} ||w||^2 + C \sum_{i=1}^{n} \xi_i$$

Subject to: $y_i(w^T x_i + b) \geq 1 - \xi_i$

Where:
- $w$: weight vector
- $b$: bias term
- $C$: regularization parameter
- $\xi_i$: slack variables

### Kernel Trick
Map data to higher dimensional space: $K(x_i, x_j) = \phi(x_i)^T \phi(x_j)$

**Common Kernels:**
1. **Linear**: $K(x_i, x_j) = x_i^T x_j$
2. **Polynomial**: $K(x_i, x_j) = (\gamma x_i^T x_j + r)^d$
3. **RBF (Gaussian)**: $K(x_i, x_j) = \exp(-\gamma ||x_i - x_j||^2)$
4. **Sigmoid**: $K(x_i, x_j) = \tanh(\gamma x_i^T x_j + r)$

## 3. Visualization Helper Functions

In [ ]:
def plot_decision_boundary(X, y, model, title, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))

    # Create decision boundary plot
    DecisionBoundaryDisplay.from_estimator(
        model, X, cmap=plt.cm.RdYlBu, alpha=0.3, ax=ax, response_method='predict'
    )

    # Plot data points
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdYlBu,
                        edgecolor='black', s=100, linewidth=1.5)

    # Plot support vectors
    if hasattr(model, 'support_vectors_'):
        ax.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
                  s=200, facecolors='none', edgecolors='green', linewidth=2.5,
                  label='Support Vectors')

    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Feature 1', fontsize=12)
    ax.set_ylabel('Feature 2', fontsize=12)
    ax.legend()

    return ax

print('Visualization helper function defined!')

## 4. Linear SVM - Linearly Separable Data

In [ ]:
# Generate linearly separable data
np.random.seed(42)
X_linear, y_linear = make_classification(
    n_samples=200, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, class_sep=2.0, random_state=42
)

# Split data
X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(
    X_linear, y_linear, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_lin_scaled = scaler.fit_transform(X_train_lin)
X_test_lin_scaled = scaler.transform(X_test_lin)

print(f'Training samples: {X_train_lin_scaled.shape[0]}')
print(f'Test samples: {X_test_lin_scaled.shape[0]}')

In [ ]:
# Train Linear SVM
svm_linear = SVC(kernel='linear', C=1.0, random_state=42)
svm_linear.fit(X_train_lin_scaled, y_train_lin)

# Predictions
y_pred_linear = svm_linear.predict(X_test_lin_scaled)
accuracy_linear = accuracy_score(y_test_lin, y_pred_linear)

print(f'Linear SVM Accuracy: {accuracy_linear:.4f}')
print(f'Number of Support Vectors: {len(svm_linear.support_vectors_)}')
print(f'Support Vectors per class: {svm_linear.n_support_}')

# Visualize
plot_decision_boundary(X_train_lin_scaled, y_train_lin, svm_linear,
                      'Linear SVM - Decision Boundary')
plt.tight_layout()
plt.show()

## 5. Effect of C Parameter (Regularization)

In [ ]:
# Test different C values
C_values = [0.01, 0.1, 1, 10, 100]
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for idx, C in enumerate(C_values):
    svm_c = SVC(kernel='linear', C=C, random_state=42)
    svm_c.fit(X_train_lin_scaled, y_train_lin)

    y_pred_c = svm_c.predict(X_test_lin_scaled)
    acc_c = accuracy_score(y_test_lin, y_pred_c)

    plot_decision_boundary(X_train_lin_scaled, y_train_lin, svm_c,
                          f'C={C}, Acc={acc_c:.3f}, SV={len(svm_c.support_vectors_)}',
                          ax=axes[idx])

# Remove extra subplot
fig.delaxes(axes[5])
plt.tight_layout()
plt.show()

print('Observation: Smaller C allows more margin violations (softer margin)')
print('            Larger C enforces stricter margin (harder margin, may overfit)')

## 6. Polynomial Kernel - Non-Linear Data

In [ ]:
# Generate polynomial-separable data
from sklearn.datasets import make_circles
X_poly, y_poly = make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=42)

X_train_poly, X_test_poly, y_train_poly, y_test_poly = train_test_split(
    X_poly, y_poly, test_size=0.2, random_state=42
)

scaler_poly = StandardScaler()
X_train_poly_scaled = scaler_poly.fit_transform(X_train_poly)
X_test_poly_scaled = scaler_poly.transform(X_test_poly)

In [ ]:
# Compare linear vs polynomial kernel
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Linear kernel (will perform poorly)
svm_lin_fail = SVC(kernel='linear', C=1.0)
svm_lin_fail.fit(X_train_poly_scaled, y_train_poly)
y_pred_lin_fail = svm_lin_fail.predict(X_test_poly_scaled)
acc_lin_fail = accuracy_score(y_test_poly, y_pred_lin_fail)

plot_decision_boundary(X_train_poly_scaled, y_train_poly, svm_lin_fail,
                      f'Linear Kernel (Acc={acc_lin_fail:.3f})', ax=axes[0])

# Polynomial kernel
svm_poly = SVC(kernel='poly', degree=3, C=1.0, gamma='auto')
svm_poly.fit(X_train_poly_scaled, y_train_poly)
y_pred_poly = svm_poly.predict(X_test_poly_scaled)
acc_poly = accuracy_score(y_test_poly, y_pred_poly)

plot_decision_boundary(X_train_poly_scaled, y_train_poly, svm_poly,
                      f'Polynomial Kernel (degree=3, Acc={acc_poly:.3f})', ax=axes[1])

plt.tight_layout()
plt.show()

print(f'Linear Kernel Accuracy: {acc_lin_fail:.4f}')
print(f'Polynomial Kernel Accuracy: {acc_poly:.4f}')

In [ ]:
# Test different polynomial degrees
degrees = [2, 3, 4, 5]
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

for idx, degree in enumerate(degrees):
    svm_deg = SVC(kernel='poly', degree=degree, C=1.0, gamma='auto')
    svm_deg.fit(X_train_poly_scaled, y_train_poly)
    y_pred_deg = svm_deg.predict(X_test_poly_scaled)
    acc_deg = accuracy_score(y_test_poly, y_pred_deg)

    plot_decision_boundary(X_train_poly_scaled, y_train_poly, svm_deg,
                          f'Polynomial Kernel (degree={degree}, Acc={acc_deg:.3f})',
                          ax=axes[idx])

plt.tight_layout()
plt.show()

## 7. RBF (Radial Basis Function) Kernel

In [ ]:
# Generate complex non-linear data
X_moons, y_moons = make_moons(n_samples=300, noise=0.15, random_state=42)

X_train_moons, X_test_moons, y_train_moons, y_test_moons = train_test_split(
    X_moons, y_moons, test_size=0.2, random_state=42
)

scaler_moons = StandardScaler()
X_train_moons_scaled = scaler_moons.fit_transform(X_train_moons)
X_test_moons_scaled = scaler_moons.transform(X_test_moons)

In [ ]:
# RBF kernel with different gamma values
gamma_values = [0.01, 0.1, 1, 10]
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

for idx, gamma in enumerate(gamma_values):
    svm_rbf = SVC(kernel='rbf', C=1.0, gamma=gamma)
    svm_rbf.fit(X_train_moons_scaled, y_train_moons)
    y_pred_rbf = svm_rbf.predict(X_test_moons_scaled)
    acc_rbf = accuracy_score(y_test_moons, y_pred_rbf)

    plot_decision_boundary(X_train_moons_scaled, y_train_moons, svm_rbf,
                          f'RBF Kernel (γ={gamma}, Acc={acc_rbf:.3f})',
                          ax=axes[idx])

plt.tight_layout()
plt.show()

print('Observation: Small gamma = broader, smoother decision boundary')
print('            Large gamma = tight fit around data points (risk of overfitting)')

## 8. Sigmoid Kernel

In [ ]:
# Sigmoid kernel
svm_sigmoid = SVC(kernel='sigmoid', C=1.0, gamma='auto')
svm_sigmoid.fit(X_train_moons_scaled, y_train_moons)
y_pred_sigmoid = svm_sigmoid.predict(X_test_moons_scaled)
acc_sigmoid = accuracy_score(y_test_moons, y_pred_sigmoid)

plot_decision_boundary(X_train_moons_scaled, y_train_moons, svm_sigmoid,
                      f'Sigmoid Kernel (Acc={acc_sigmoid:.3f})')
plt.tight_layout()
plt.show()

print(f'Sigmoid Kernel Accuracy: {acc_sigmoid:.4f}')

## 9. Comprehensive Kernel Comparison

In [ ]:
# Compare all kernels on moons dataset
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

results_kernels = {}

for idx, kernel in enumerate(kernels):
    if kernel == 'poly':
        svm = SVC(kernel=kernel, degree=3, C=1.0, gamma='auto')
    else:
        svm = SVC(kernel=kernel, C=1.0, gamma='auto')

    svm.fit(X_train_moons_scaled, y_train_moons)
    y_pred = svm.predict(X_test_moons_scaled)
    acc = accuracy_score(y_test_moons, y_pred)
    results_kernels[kernel] = acc

    plot_decision_boundary(X_train_moons_scaled, y_train_moons, svm,
                          f'{kernel.upper()} Kernel (Acc={acc:.3f})',
                          ax=axes[idx])

plt.tight_layout()
plt.show()

# Bar plot comparison
plt.figure(figsize=(10, 6))
plt.bar(results_kernels.keys(), results_kernels.values(),
        color=['skyblue', 'lightgreen', 'lightcoral', 'gold'])
plt.xlabel('Kernel Type', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('SVM Kernel Comparison on Moons Dataset', fontsize=14, fontweight='bold')
plt.ylim([0.5, 1.0])
for i, (kernel, acc) in enumerate(results_kernels.items()):
    plt.text(i, acc + 0.02, f'{acc:.4f}', ha='center', fontsize=11, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Real-World Dataset - Breast Cancer

In [ ]:
# Load Breast Cancer dataset
cancer = load_breast_cancer()
X_cancer = cancer.data
y_cancer = cancer.target

X_train_cancer, X_test_cancer, y_train_cancer, y_test_cancer = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer
)

scaler_cancer = StandardScaler()
X_train_cancer_scaled = scaler_cancer.fit_transform(X_train_cancer)
X_test_cancer_scaled = scaler_cancer.transform(X_test_cancer)

print(f'Dataset: {cancer.DESCR.split(chr(10))[0]}')
print(f'Features: {X_cancer.shape[1]}')
print(f'Samples: {X_cancer.shape[0]}')

In [ ]:
# Train SVM with different kernels
kernels_cancer = ['linear', 'poly', 'rbf', 'sigmoid']
results_cancer = {}

for kernel in kernels_cancer:
    if kernel == 'poly':
        svm = SVC(kernel=kernel, degree=3, C=1.0, gamma='scale', random_state=42)
    else:
        svm = SVC(kernel=kernel, C=1.0, gamma='scale', random_state=42)

    svm.fit(X_train_cancer_scaled, y_train_cancer)
    y_pred = svm.predict(X_test_cancer_scaled)

    acc = accuracy_score(y_test_cancer, y_pred)
    f1 = f1_score(y_test_cancer, y_pred)

    results_cancer[kernel] = {'accuracy': acc, 'f1': f1}

    print(f'\n{kernel.upper()} Kernel:')
    print(f'  Accuracy: {acc:.4f}')
    print(f'  F1-Score: {f1:.4f}')
    print(f'  Support Vectors: {len(svm.support_vectors_)}')

In [ ]:
# Visualize performance
df_results = pd.DataFrame(results_cancer).T

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_results['accuracy'].plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Accuracy by Kernel', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].set_xlabel('Kernel', fontsize=11)
axes[0].set_xticklabels(df_results.index, rotation=45)
axes[0].grid(axis='y', alpha=0.3)

df_results['f1'].plot(kind='bar', ax=axes[1], color='lightcoral', edgecolor='black')
axes[1].set_title('F1-Score by Kernel', fontsize=13, fontweight='bold')
axes[1].set_ylabel('F1-Score', fontsize=11)
axes[1].set_xlabel('Kernel', fontsize=11)
axes[1].set_xticklabels(df_results.index, rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Hyperparameter Tuning with Grid Search

In [ ]:
# Grid search for optimal hyperparameters
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'kernel': ['rbf', 'poly']
}

grid_search = GridSearchCV(SVC(), param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train_cancer_scaled, y_train_cancer)

print(f'Best parameters: {grid_search.best_params_}')
print(f'Best cross-validation score: {grid_search.best_score_:.4f}')

# Test on test set
best_svm = grid_search.best_estimator_
y_pred_best = best_svm.predict(X_test_cancer_scaled)
best_accuracy = accuracy_score(y_test_cancer, y_pred_best)
print(f'Test set accuracy: {best_accuracy:.4f}')

In [ ]:
# Confusion matrix for best model
cm = confusion_matrix(y_test_cancer, y_pred_best)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=cancer.target_names, yticklabels=cancer.target_names)
plt.title('Confusion Matrix - Best SVM Model', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_test_cancer, y_pred_best, target_names=cancer.target_names))

## 12. Multi-class Classification - Iris Dataset

In [ ]:
# Load Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

scaler_iris = StandardScaler()
X_train_iris_scaled = scaler_iris.fit_transform(X_train_iris)
X_test_iris_scaled = scaler_iris.transform(X_test_iris)

# Train multi-class SVM
svm_multiclass = SVC(kernel='rbf', C=10, gamma='scale', decision_function_shape='ovr')
svm_multiclass.fit(X_train_iris_scaled, y_train_iris)

y_pred_iris = svm_multiclass.predict(X_test_iris_scaled)
acc_iris = accuracy_score(y_test_iris, y_pred_iris)

print(f'Multi-class SVM Accuracy: {acc_iris:.4f}')

# Confusion matrix
cm_iris = confusion_matrix(y_test_iris, y_pred_iris)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_iris, annot=True, fmt='d', cmap='Greens',
            xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.title('Multi-class SVM - Iris Dataset', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

## 13. Key Takeaways

### SVM Fundamentals
- **Goal**: Find optimal hyperplane that maximizes margin between classes
- **Support Vectors**: Data points closest to decision boundary
- **Margin**: Distance between hyperplane and nearest data points

### Hyperparameters
#### C (Regularization)
- **Small C**: Soft margin, more tolerance for misclassifications
- **Large C**: Hard margin, less tolerance (may overfit)

#### Gamma (for RBF, Poly, Sigmoid)
- **Small gamma**: Broader influence, smoother decision boundary
- **Large gamma**: Tight influence, complex boundary (may overfit)

### Kernel Selection Guide
1. **Linear Kernel**
   - Use when: Data is linearly separable or high-dimensional
   - Advantages: Fast, interpretable
   - Examples: Text classification, high-dimensional sparse data

2. **Polynomial Kernel**
   - Use when: Features have polynomial relationships
   - Hyperparameters: degree, gamma, coef0
   - Note: Can be computationally expensive

3. **RBF (Gaussian) Kernel**
   - Use when: No prior knowledge of data, general-purpose
   - Advantages: Works well for most non-linear problems
   - Most popular choice for non-linear data

4. **Sigmoid Kernel**
   - Use when: Similar to neural network activation
   - Less common in practice
   - Can be unstable with some parameter values

### Advantages of SVM
- Effective in high-dimensional spaces
- Memory efficient (uses subset of training points)
- Versatile (different kernels)
- Works well with clear margin of separation

### Limitations
- Slow training for large datasets
- Sensitive to feature scaling
- Requires careful hyperparameter tuning
- No probabilistic interpretation (without calibration)
- Black box for non-linear kernels

### Best Practices
1. Always scale/normalize features
2. Start with RBF kernel for non-linear data
3. Use cross-validation for hyperparameter tuning
4. Consider linear kernel for high-dimensional data
5. Balance C and gamma to avoid overfitting